In [ ]:

# SMART CITY – POLLUTION PIPELINE 


import requests
import pandas as pd
import mysql.connector
from datetime import datetime


# API CONFIG


API_URL = (
    #"insert api url here"
)

response = requests.get(API_URL)

if response.status_code != 200:
    print("API Error")
    exit()

data = response.json()
hourly = data["hourly"]

df = pd.DataFrame({
    "datetime": hourly["time"],
    "pm25": hourly["pm2_5"],
    "pm10": hourly["pm10"],
    "carbon_monoxide": hourly["carbon_monoxide"],
    "nitrogen_dioxide": hourly["nitrogen_dioxide"],
    "sulphur_dioxide": hourly["sulphur_dioxide"],
    "ozone": hourly["ozone"]
})

df["city"] = "Delhi"
df["datetime"] = pd.to_datetime(df["datetime"])


# STRUCTURED TIME LOGIC


current_time = datetime.utcnow().replace(minute=0, second=0, microsecond=0)

def classify_time(row_time):
    diff = row_time - current_time
    hours_diff = int(diff.total_seconds() / 3600)

    if hours_diff < 0:
        return 1, 0, 0, hours_diff
    elif hours_diff == 0:
        return 0, 1, 0, 0
    else:
        return 0, 0, 1, hours_diff

df[["is_past", "is_present", "is_future", "hours_from_now"]] = df["datetime"].apply(
    lambda x: pd.Series(classify_time(x))
)


# MYSQL CONNECTION


conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="",  # use your password here
    database="smart_city"
)

cursor = conn.cursor()

# Clear old data
cursor.execute("TRUNCATE TABLE pollution_delhi")
conn.commit()

# Insert data
insert_query = """
INSERT INTO pollution_delhi
(datetime, pm25, pm10, carbon_monoxide, nitrogen_dioxide, sulphur_dioxide, ozone, city,
 is_past, is_present, is_future, hours_from_now)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

for _, row in df.iterrows():
    cursor.execute(insert_query, tuple(row))

conn.commit()

print("✅ Pollution data inserted successfully!")

cursor.close()
conn.close()


✅ Pollution data inserted successfully!
